# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library. The dataset includes tabular clinical and molecular data for cancer survivors with second primary colorectal cancer.

### Dataset Source
- Croissant schema URL: [`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

# Install matplotlib for visualization later
!pip install matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nVersion: {metadata.version}")

## 2. Data Overview
This section reviews available record sets and their fields. All record sets and fields are referenced by their `@id`.

Let's list all available record sets and their fields by `@id`.

In [ ]:
# Helper to print record sets and fields by @id

record_sets = metadata.record_sets
print("Available record sets and their fields:")
for rs in record_sets:
    print(f"- Record set name: {getattr(rs, 'name', 'N/A')} (@id: {rs.id})")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - Field: {getattr(field, 'name', 'N/A')} (@id: {field.id}) - type: {getattr(field, 'data_type', 'N/A')}")

To get a sense of the data, let's view a sample record from each record set.

In [ ]:
# Print a sample record from each record set
for rs in record_sets:
    print(f"\nRecord set: {getattr(rs, 'name', 'N/A')} (@id: {rs.id})")
    try:
        recs = dataset.records(record_set=rs.id)
        sample = next(recs, None)
        if sample:
            print(sample)
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

## 3. Data Extraction
Load all records from the main record set(s) into pandas DataFrames for analysis.

> All record sets and fields are referenced **by their `@id`** as required by the Croissant specification.

In [ ]:
# Identify main tabular record set(s) by @id
main_rs_ids = [rs.id for rs in record_sets if hasattr(rs, 'fields') and len(rs.fields) > 0]
print("Record set IDs with tabular data:")
for rsid in main_rs_ids:
    print(f"  {rsid}")

dataframes = {}
for record_set_id in main_rs_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for Record Set {record_set_id}: columns=")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let’s process the main record set. We'll:

- Select a **numeric field** by its `@id` (as shown in the metadata above),
- Filter records by a threshold,
- Normalize that numeric field,
- Group by a **categorical field** (if available),
- All referencing fields **by their exact `@id`**.

Replace the field IDs below with those found in your dataset if different.

In [ ]:
# For demonstration, let's pick the first tabular record set.
main_record_set_id = main_rs_ids[0]
df = dataframes[main_record_set_id].copy()

# Let's list all available field @id's in this record set:
print("Fields/Columns (@id) in main record set:")
for col in df.columns:
    print(f"- {col}")

# Try to locate a numeric field (e.g., 'Age' or a field containing 'age', 'interval', etc.)
import re
numeric_candidates = [c for c in df.columns if re.search(r'age|interval|count|number|metastasis|size|score', c, re.I)]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    raise ValueError("No numeric field candidates found; please adjust field selection.")

# Set a threshold for filtering
threshold = df[numeric_field_id].dropna().median()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

normalized_field = f"{numeric_field_id}_normalized"
filtered_df[normalized_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, normalized_field]].head())

# Attempt to find a categorical/group field (e.g., 'sex', 'gender', or 'status', or use any non-numeric field)
categorical_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
if categorical_candidates:
    group_field_id = categorical_candidates[0]
    print(f"\nGrouping by: {group_field_id}")
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped.head())
else:
    group_field_id = None
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to a categorical variable (if available).


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
filtered_df[numeric_field_id].hist(bins=15)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field_id} in filtered records')
plt.show()

if group_field_id:
    plt.figure(figsize=(8,5))
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- This walkthrough showed how to load, inspect, and analyze a Croissant-compliant dataset using `mlcroissant`.
- All references to data structures use their Croissant `@id` identifiers for clarity and reproducibility.
- You can repeat the EDA steps for additional record sets and fields to continue exploring the dataset.

> **Tip:** For more details about any field or record set, refer to the Croissant schema or use `dataset.metadata` attributes.